<a id="top"></a>

# Demo: the single-step task catalog

```yaml
title: "Demo: the single-step task catalog"
keywords: task shapes, extraction, classification, ranking, constrained generation, summarization, structured output, teacher demo
estimated duration: 15
```

> **Lesson:** L02. This is Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L02/demos_or_activities.md). It makes **live** Claude calls through the `potato_llm` seam -- set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies** run to run, so a given run may not show every failure the prose calls out -- that variance is why every shape ends in a **validator**. Clear outputs before committing.
>
> One toolkit, five shapes: each task below is just **roles + structured output + few-shot** aimed at a different **output contract**. Name the shape, ask for its contract, **validate** the contract.

## Contents

- [1. Setup](#1-setup)
- [2. Extraction](#2-extraction)
- [3. Classification](#3-classification)
- [4. Ranking](#4-ranking)
- [5. Constrained generation](#5-constrained-generation)
- [6. Summarization](#6-summarization)
- [7. Takeaways](#7-takeaways)

## 1. Setup

You've got one live client, one defensive JSON parser (the same discipline as the structured-output demo), and one `ask` helper. Every shape below reuses them.

In [ ]:
import json
import re
from typing import Any

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

# Live client. The key is read through fluffy_potato_curriculum.common.config
# (a pydantic-settings layer over the environment / .env); constructing the client
# raises a clear error if ANTHROPIC_API_KEY is missing, rather than failing mid-call.
client: PotatoLLMClient = AnthropicClient()


def parse_json(reply: str) -> Any:
    """Defensive JSON parse: whole reply, then salvage the first {...}/[...] block, then fail loudly.

    Reused from the structured-output demo -- every JSON-returning shape below leans on it.
    """
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        pass
    match = re.search(r"[\[{].*[\]}]", reply, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass
    raise ValueError(f"Could not parse JSON from reply:\n{reply!r}")


async def ask(system: str, user: str, *, max_tokens: int = 300, temperature: float = 0.0) -> str:
    """One live call: a system policy + user content -> the raw reply text."""
    reply = await client.achat(
        [Message.system(system), Message.user(user)],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return reply.text


print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Extraction

**Extraction = structured output pointed at "pull these fields."** You can ask for a fixed schema every time, or a *mixed* schema that pulls a list of heterogeneous items. **Validate:** the required keys are present. Watch for dropped or invented fields.

In [ ]:
# Fixed schema: the same keys every time.
EMAIL = "Hi, this is Dana Lopez. My order A-4471 arrived broken and I'd like a refund."
fixed = await ask(
    "Extract customer_name, order_id, and intent (one of: refund, exchange, question, "
    "complaint) from the email. Return one JSON object with exactly those keys, no prose.",
    EMAIL,
)
print("fixed schema :", parse_json(fixed))

# Mixed schema: a list of heterogeneous line-items, each validated against the shape it claims.
RECEIPT = "2x Widget SKU-88 at $4 each; promo SAVE10 took 10% off; 1x Gadget SKU-90 at $20."
mixed = await ask(
    "Extract each line item from the receipt as a JSON array. A product item is "
    '{"kind": "product", "sku": ..., "qty": ...}; a discount item is '
    '{"kind": "discount", "code": ..., "percent": ...}. Return only the JSON array.',
    RECEIPT,
)
print("mixed schema :", parse_json(mixed))

[↑ Back to top](#top)

## 3. Classification

**Classification = structured output constrained to a label set** -- and this is where **few-shot** earns its keep, because label *wording* is often idiosyncratic. You can go flat, hierarchical (category -> subcategory), or multi-label. **Validate:** the label is in the allowed set.

In [ ]:
TICKET = "The checkout page returns a 500 error for every user since the deploy."
LABELS = ["P0-bug", "feature-discussion", "docs-clarity", "billing-question", "thank-you-note"]

# Flat: exactly one label from the set.
flat = (
    await ask(
        "Classify the ticket into exactly one of: "
        + ", ".join(LABELS)
        + ". Return only the label.",
        TICKET,
    )
).strip()
print("flat label :", repr(flat), "| in allowed set:", flat in LABELS)

# Hierarchical taxonomy: a category plus a refining subcategory.
taxo = await ask(
    'Classify the ticket. Return a JSON object {"category": ..., "subcategory": ...} where '
    "category is one of [bug, request, question].",
    TICKET,
)
print("taxonomy   :", parse_json(taxo))

[↑ Back to top](#top)

## 4. Ranking

**Give the candidates ids; ask for the ids back in order** -- never let the model rewrite the candidates. **Validate:** every id appears exactly once (no drops, dupes, or inventions). This is the classic silent failure.

In [ ]:
CANDIDATES = [
    {"id": 1, "text": "Add dark mode."},
    {"id": 2, "text": "Fix data loss on logout."},
    {"id": 3, "text": "Support CSV export."},
    {"id": 4, "text": "Translate the UI to French."},
]
ranked = parse_json(
    await ask(
        "Rank the feature requests by likely user impact, most important first. "
        "Return only a JSON array of their id numbers -- nothing else.",
        json.dumps(CANDIDATES),
    )
)
print("ranked ids :", ranked)

# Validate the contract: the id set is preserved exactly (no drops, dupes, or inventions).
expected = {c["id"] for c in CANDIDATES}
problems: list[str] = []
if len(ranked) != len(set(ranked)):
    problems.append("duplicate ids")
if set(ranked) != expected:
    problems.append(f"id set changed: got {sorted(set(ranked))}, expected {sorted(expected)}")
print("contract   :", problems or "ok -- every candidate ranked exactly once")

[↑ Back to top](#top)

## 5. Constrained generation

Generation is the one shape where you may want a **higher temperature** -- but the **constraint** (a count, a length cap) must be **explicit and re-checked in code**. Ask for a JSON array of length N so you can `len()`-check it. **Validate:** exactly N items, each within the bound.

In [ ]:
N, MAX_WORDS = 5, 8
generated = parse_json(
    await ask(
        f"Generate exactly {N} onboarding-email subject lines, each at most {MAX_WORDS} words. "
        "Return only a JSON array of strings.",
        "Product: a habit-tracking mobile app for students.",
        temperature=0.7,  # some variety is wanted here
    )
)
print("generated  :", generated)

# Validate the contract: exactly N items, each within the word bound.
problems = []
if len(generated) != N:
    problems.append(f"expected {N} items, got {len(generated)}")
for line in generated:
    if len(line.split()) > MAX_WORDS:
        problems.append(f"over {MAX_WORDS} words: {line!r}")
print("contract   :", problems or f"ok -- exactly {N}, each <= {MAX_WORDS} words")

[↑ Back to top](#top)

## 6. Summarization

**Summarize / rewrite / normalize / translate** are one family: text in, transformed text out, **meaning preserved**. The **system message** carries the durable policy (audience, length, register); the user text is what changes. **Validate** with guardrails, not equality: length, no invented facts.

In [ ]:
DOC = (
    "The Q3 release shipped offline mode, cut cold-start time by 40%, and fixed the sync bug "
    "that occasionally duplicated tasks. Offline-mode adoption hit 30% in the first two weeks."
)

# The system policy fixes audience + length; swap it to restyle without touching the user text.
summary = await ask("Summarize the update in one sentence for a non-technical executive.", DOC)
rewrite = await ask("Rewrite the update as an upbeat one-line changelog entry with an emoji.", DOC)
print("summary :", summary.strip())
print("rewrite :", rewrite.strip())

[↑ Back to top](#top)

## 7. Takeaways

- **One toolkit, five shapes.** Extraction, classification, ranking, constrained generation, and summarization are all **roles + structured output + few-shot** aimed at a different **output contract** -- not a new technique per task.
- **The contract is the thing you *validate*,** not just the thing you ask for: required keys (extraction), label-in-set (classification), no-drop/no-dupe (ranking), exact count + bound (constrained generation), guardrails (summarization).
- **Temperature by shape** (L01): near-0 for extraction / classification / ranking; lift it only where generation wants variety.
- **Each shape is one *node* in disguise.** L03 wraps extraction as a graph node; L03--L05 chain shapes into a pipeline.

[↑ Back to top](#top)